Script Overview:
1. Excluded words are combined with spaCys default stop words
2. Docs are tokenized from line 4 (excluding date and title) and words are excluded
3. Words with a frequency of one or two are removed and terms occuring in more than 95% of the docs are removed
4. The tokenized text is prepared for topic modeling and converted to a BOW
5. Hyperparameter optimization is performed to find the best parameters for the LDA model
6. LDA model is run with best parameters
7. Topics are visualized 
8. Top 5 words per topic are printed
9. Uncertainty terms calculated per doc
10. Number of docs per topic are calculated through the probability distribution given for each document from LDA (e.g if 50% of doc 1 is topic 2 then topic 2 gets given 0.5)
11. Number of uncertainty terms per topic is calculated the same way (e.g. doc 1 has 5 uncertainty terms, 50% of doc 1 is topic 2 so topic 2 gets assigned 2.5 words)
12. Chi square goodness of fit assumption check, if met test is performed if not permutation test is performed. 


1. Excluded words are combined with spaCys default stop words

In [3]:
excluded_words = ["de", "in", "aan", "als", "een", "van", "en", "van", "een", "door", "voor", "hun", "hen", "CBS", "dan", "was", "waren"
                  , "dit", "of", "is", "het", "er", "bij", "tot", "procent", "jaar", "naar", 
                  "deze", "niet", "cbs", "te", "over", 
                  "die", "hebben", "met", " ", "dat", "op", "had", "maar", "in", "zijn", "ook", "$", "uit", "per"
                  , "onder", "article", "nederland", "kwartaal", "aantal", "nog", "duizend", "ruim", "title",
                   "eerste", "vooral", "content", "text", "lead", "alle", "tweede", "om", "toe", "cijfers", "ten",
                   "miljoen", "euro", "mensen", "nam", "derde", "worden", "nederlandsen", "miljard", "gemiddeld",
                   "wordt", "opzichte", "omzet", "bedrijven", "industrie", "veel", "weinig", "hoog", "eerder", "groot",
                   "één", "komen", "laag", "twee", "mens", "drie", "al","maand", "vaak", "nemen", "stijgen",
                   "dalen", "cijfer", "nederlands", "gaan", "opzicht", "ander", "nieuw", "tussen", "vier", "goed",
                   "totaal", "sterk", "kunnen"]
excluded_words.extend(nlp.Defaults.stop_words)
excluded_words = list(set(excluded_words))

2. Docs are tokenized from line 4 (excluding date and title) and words are excluded

In [4]:
import os
import spacy
from nltk.util import ngrams
from sklearn.feature_extraction.text import CountVectorizer


nlp = spacy.load("nl_core_news_sm")


# tokenization fnc
excluded_tokens = set()
def tokenize_text(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()[4:] # from lead text 
        text = "".join(lines)
    doc = nlp(text)
    words = []
    for token in doc:
        if token.is_alpha and token.lemma_.lower() not in excluded_words: 
            words.append(token.lemma_.lower())
        else:
            excluded_tokens.add(token.text.lower())

    return " ".join(words)



# Directory with files
directory = "filt_art"  
docs = []
filenames = []
for file in os.listdir(directory):
    file_path = os.path.join(directory, file)
    token_text = tokenize_text(file_path)
    docs.append(token_text)
    filenames.append(file)

vectorizer = CountVectorizer(stop_words = excluded_words)

# outputs a sparse matrix
NL_matrix = vectorizer.fit_transform(docs)

3. Words with a frequency of one or two are removed and terms occuring in more than 95% of the docs are removed

In [5]:
import pandas as pd
nl_df = pd.DataFrame(NL_matrix.toarray(), index = filenames, columns = vectorizer.get_feature_names_out())
n_totals_1 = nl_df.columns[nl_df.sum(axis = 0)==1]
nl_df = nl_df.drop(columns = n_totals_1)
n_totals_2 = nl_df.columns[nl_df.sum(axis = 0)==2]
nl_df = nl_df.drop(columns = n_totals_2)
term_counts = (nl_df > 0).sum(axis = 0)

# getting terms that occur in more than 95% of docs
terms_95 = nl_df.columns[term_counts > 0.95*len(nl_df)]

nl_df = nl_df.drop(columns= terms_95)

4. The tokenized text is prepared for topic modeling and converted to a BOW

In [ ]:
# load packages for tokenization
import gensim
from scipy.sparse import csr_matrix
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [8]:
dictionary = Dictionary([[word] for word in nl_df.columns])
bow = []
for idx, row in nl_df.iterrows():
    doc_bow = [(dictionary.token2id[word], count) for word, count in row.items() if count > 0]
    bow.append(doc_bow)

5. Hyperparameter optimization is performed to find the best parameters for the LDA model

In [11]:
import numpy as np

num_topics_range = [17,19,20,22,25]
alpha =["auto","symmetric", "asymmetric", 0.1,0.3,0.5, 0.8]
eta = ["auto","symmetric", 0.01, 0.03, 0.05, 0.08, 0.1, 0.5, 0.7, 0.9]


def coherence_score_fnc(model, texts, dictionary):
    cv_model = CoherenceModel(model = model, texts = texts, dictionary = dictionary, coherence= "c_v")
    cv_score = cv_model.get_coherence()
    return cv_score

texts = [doc.split() for doc in docs]

best_coherence = -1
best_model = None
best_params = None


# set a seed for reproducibility
seed = 42

scores = []

for num_topics in num_topics_range:
    for a in alpha:
        for e in eta:
            print(f"Model with topics:{num_topics}, alpha:{a}, eta:{e}")

            lda_model = gensim.models.LdaModel(bow, num_topics= num_topics, id2word = dictionary, passes= 10, alpha = a, eta = e, random_state= seed)

            coherence_score = coherence_score_fnc(model = lda_model, texts = texts, dictionary= dictionary)
            print(f"Coherence Score: {coherence_score}")

            scores.append((coherence_score, num_topics, a ,e))

        

scores.sort(reverse = True, key = lambda x:x[0])
for i in range(min(5, len(scores))):
    print(f"Position {i + 1}: Coherence Score = {scores[i][0]}, Parameters: num_topics = {scores[i][1]}, alpha = {scores[i][2]}, eta = {scores[i][3]}")

Model with topics:17, alpha:auto, eta:auto
Coherence Score: 0.4587622177641555
Model with topics:17, alpha:auto, eta:symmetric
Coherence Score: 0.4587622177641555
Model with topics:17, alpha:auto, eta:0.01
Coherence Score: 0.46326070161094685
Model with topics:17, alpha:auto, eta:0.03
Coherence Score: 0.46338285237225463
Model with topics:17, alpha:auto, eta:0.05
Coherence Score: 0.4599353728141461
Model with topics:17, alpha:auto, eta:0.08
Coherence Score: 0.4560345514254102
Model with topics:17, alpha:auto, eta:0.1
Coherence Score: 0.45448715314794846
Model with topics:17, alpha:auto, eta:0.5
Coherence Score: 0.4486191698939963
Model with topics:17, alpha:auto, eta:0.7
Coherence Score: 0.4588045643359927
Model with topics:17, alpha:auto, eta:0.9
Coherence Score: 0.4416224629882542
Model with topics:17, alpha:symmetric, eta:auto
Coherence Score: 0.4587622177641555
Model with topics:17, alpha:symmetric, eta:symmetric
Coherence Score: 0.4587622177641555
Model with topics:17, alpha:symme

6. LDA model is run with best parameters

In [9]:
# LDA Model with best parameters:
import numpy as np
lda = gensim.models.LdaModel(
    bow, num_topics=19, id2word=dictionary, passes=10, alpha= 0.8, eta=0.1, random_state= 42
)

coherence_model = CoherenceModel(model= lda, texts = [doc.split() for doc in docs] , dictionary= dictionary,
                              coherence = "c_v")
with np.errstate(invalid= "ignore"): 
    c_cv = coherence_model.get_coherence()  

#print measure
c_cv 

0.5287339936666191

7. Topics are visualized 

In [10]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
visual = gensimvis.prepare(lda, bow, dictionary)
pyLDAvis.display(visual)

8. Top 5 words per topic are printed

In [11]:
for i, topic in enumerate(lda.show_topics(num_topics=20, num_words=5, formatted=False)):
    print(f"Topic {i+1}:")
    for word, weight in topic[1]:
        print(f"  {word}: {weight}")

Topic 1:
  jong: 0.033883512020111084
  werken: 0.014468695968389511
  inwoner: 0.012986505404114723
  vrouw: 0.011048758402466774
  man: 0.010913846082985401
Topic 2:
  werknemer: 0.052947692573070526
  zorg: 0.01934569887816906
  werk: 0.014627582393586636
  welzijn: 0.014302329160273075
  werken: 0.012798610143363476
Topic 3:
  jong: 0.02672753483057022
  krijgen: 0.01989598199725151
  nareiziger: 0.010840299539268017
  asielzoeker: 0.009653689339756966
  helft: 0.008444591425359249
Topic 4:
  vervoeren: 0.023007575422525406
  passagier: 0.0182707067579031
  bijstand: 0.017043422907590866
  airport: 0.016573533415794373
  ton: 0.014782460406422615
Topic 5:
  ondernemer: 0.03458184376358986
  bedrijf: 0.02048606052994728
  begin: 0.01813834346830845
  bedrijfstak: 0.016695424914360046
  verwachten: 0.013674324378371239
Topic 6:
  inflatie: 0.05139339342713356
  cpi: 0.03571930527687073
  hicp: 0.025556275621056557
  prijs: 0.01980196312069893
  prijsontwikkeling: 0.017092274501919746

9. Uncertainty terms calculated per doc

In [10]:
# for this we will use Counter
import os
import pandas as pd
import spacy
from collections import Counter 
nlp = spacy.load("nl_core_news_sm")



uncertainty_terms = set([
    "altijd", "kans", "meestal", "misschien", "mogelijk", "nooit", "onmogelijk", "onwaarschijnlijk", "onzeker", 
    "soms", "twijfelachtig", "vaak", "vermoedelijk","waarschijnlijk", "zeker", "zelden", "bijna"
])

def tokenize(file):
    with open(file, "r") as f:
        lines = f.readlines()[4:]
        text = "".join(lines)
    doc = nlp(text)

    return doc

def find_uncertainty_terms(doc_tokens):
    found_terms = []
    found_terms += [w.text.lower() for w in doc_tokens if w.text.lower() in uncertainty_terms]
    return found_terms

directory = "filt_art"
tokenize_docs = []
filenames = []

for file in os.listdir(directory):
    file_path = os.path.join(directory, file)
    doc = tokenize(file_path)
    filenames.append(file)
    tokenize_docs.append(doc)    

    

10. Number of docs per topic are calculated through the probability distribution given for each document from LDA (e.g if 50% of doc 1 is topic 2 then topic 2 gets given 0.5)
11. Number of uncertainty terms per topic is calculated the same way (e.g. doc 1 has 5 uncertainty terms, 50% of doc 1 is topic 2 so topic 2 gets assigned 2.5 words)

In [11]:
# Initialize topic uncertainty counts with weights
topic_u_weighted_counts = {i:0 for i in range(lda.num_topics)}
topic_doc_counts = {i:0 for i in range(lda.num_topics)}

for idx, content in enumerate(bow):
    doc_topic_dist = lda.get_document_topics(content)
    doc_tok = tokenize_docs[idx]
 
    found_terms = find_uncertainty_terms(doc_tok)
    num_uncertainty_terms = len(found_terms)
    
    
    for topic, prob in doc_topic_dist:
        topic_doc_counts[topic] += prob
        weighted_uncertainty = num_uncertainty_terms * prob
        topic_u_weighted_counts[topic] += weighted_uncertainty


topic_u_weighted_counts


{0: 224.49848323874176,
 1: 209.98183208424598,
 2: 97.84139345958829,
 3: 129.90289126895368,
 4: 234.17804194707423,
 5: 45.07921367324889,
 6: 203.3904672311619,
 7: 54.80102496501058,
 8: 261.16150620393455,
 9: 109.90621949080378,
 10: 375.3858562996611,
 11: 133.4868266293779,
 12: 96.58121354877949,
 13: 150.73477411456406,
 14: 94.95092633739114,
 15: 91.05658560339361,
 16: 64.61837780661881,
 17: 113.08378147240728,
 18: 151.23613654077053}

In [ ]:
data = {"Topic": list(topic_u_weighted_counts.keys()),
        "Uncertainty Terms": list(topic_u_weighted_counts.values()),
        "Documents": list(topic_doc_counts.values())
        }


df =pd.DataFrame(data)
# new column calculated to account for number of docs 
df["Relative Freq"] = df["Uncertainty Terms"]/df["Documents"]

12. Chi square goodness of fit assumption check, if met test is performed if not permutation test is performed. 

In [ ]:
# E < 5 so assumption is not met
obs = df["Relative Freq"]
exp = [sum(obs)/len(obs)]*len(obs)
exp


[2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575,
 2.0692324455486575]

In [15]:
obs_values = df["Relative Freq"]
obs_var = np.var(obs_values)

n_perm = 10000
perm_vars = []

for i in range(n_perm):
    shuff_values = np.random.permutation(obs_values)
    perm_var = np.var(shuff_values)
    perm_vars.append(perm_var)


p_value = np.sum(np.array(perm_vars)>= obs_var)/n_perm

p_value

0.9765

In [16]:
obs_var

0.7393710396073543